In [ ]:
%pip install qiskit matplotlib qiskit[quantum_info] numpy scipy[optimize]

# 🔬 Notebook 7: Intro to Quantum Machine Learning

### *Ode to Quantum: Meridian Station Quantum Core Lab*

> A preview mission: the door to where this course is headed.

---

## 🛰️ Mission Briefing


Meridian Station's deep-space telescope array has picked up a repeating signal
pattern nobody recognizes. *"We have a handful of confirmed 'known' signals and a
handful of confirmed 'unknown' ones,"* ECHO says, pulling up a scatter of data
points. *"Before we wake the science team, I want a first-pass classifier; built
on a qubit, trained on this data, right now. It won't be sophisticated. But it'll
be real, and it'll show you exactly where the rest of this course is taking you."*

**A note before you start:** this notebook is a deliberate preview, not the full
Quantum Machine Learning curriculum. It skips ahead of the quantum algorithms and
variational-computing notebooks that would normally come first (they're listed in
the roadmap at the end). Treat this as a taste of the destination, not the whole
journey.

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- Explain, in plain terms, what training a machine learning model means
- Encode classical data onto a qubit using a quantum feature map
- Build a small parameterized (trainable) quantum circuit
- Train that circuit's parameter against labeled data using a classical optimizer
- Evaluate whether the trained circuit classifies new data correctly


## 🧩 Prerequisites

- Notebook 2: gates and measurement
- Notebook 3: superposition and the Ry rotation


## 💡 Concept


**The machine learning vocabulary, briefly:**

- **Features:** the input data describing each example (here: a single number per signal).
- **Labels:** the correct answer for each example (here: "known" = 0, "unknown" = 1).
- **Training:** adjusting a model's internal parameters so its predictions match
  the labels as closely as possible, using already-labeled examples.
- **Prediction:** using the trained model on *new* data whose label you don't
  already know.

**What makes this quantum:**

Instead of a classical model (like a neural network), the "model" here is a tiny
quantum circuit with two parts:

1. A **feature map:** a fixed way of encoding a classical number onto a qubit.
   We'll use **angle encoding**: turn the input number directly into a rotation
   angle with `Ry`.
2. An **ansatz:** a second, *trainable* rotation whose angle isn't fixed by the
   data. This is the part training actually adjusts.

We read out a prediction by measuring the expectation value of the qubit (a number
between -1 and +1, related to the probabilities you computed in Notebook 3) and
comparing it to the target label.

This loop: quantum circuit computes something, classical optimizer adjusts a
parameter, repeat; is called a **hybrid quantum-classical** algorithm, and it's the
same basic pattern behind VQE and QAOA, two algorithms further along in the full
roadmap.

## 📊 Visualization


Here's our toy "telescope signal" dataset: twelve signals, each described by a
single number, labeled `0` (known) or `1` (unknown).


## 🧪 Hands-on Code


First, build (and look at) the dataset.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
xs = np.concatenate([rng.normal(-1.2, 0.4, 6), rng.normal(1.2, 0.4, 6)])
ys = np.array([0]*6 + [1]*6)  # 0 = known signal, 1 = unknown signal

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.scatter(xs, np.zeros_like(xs), c=ys, cmap="coolwarm", s=80, edgecolor="k")
ax.set_yticks([])
ax.set_xlabel("signal feature value")
ax.set_title("Known (blue) vs. Unknown (red) telescope signals")
fig.tight_layout()


Now build the quantum model: one qubit, one feature-map rotation (fixed by the
data), one ansatz rotation (trainable).


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

def model_circuit(x, theta):
    qc = QuantumCircuit(1)
    qc.ry(x, 0)         # feature map: encode the data point
    qc.ry(theta[0], 0)  # ansatz: the ONE trainable parameter
    return qc

# What does an untrained model even look like?
print(model_circuit(xs[0], [0.0]).draw())


In [ ]:
Z = SparsePauliOp("Z")

def predict_score(x, theta):
    """Return the expectation value of Z: a number in [-1, +1]."""
    sv = Statevector(model_circuit(x, theta))
    return sv.expectation_value(Z).real

# Before any training, try a starting guess
theta_start = [0.1]
print("Sample scores before training:", [round(predict_score(x, theta_start), 2) for x in xs[:4]])


## 📐 Math Lens


We need a way to compare the model's output to the labels. Labels are `0`/`1`; the
model's output is an expectation value in `[-1, +1]`. We convert the label to the
same scale with $1 - 2y$ (so label `0` $\to$ target `+1`, label `1` $\to$ target `-1`), then
use ordinary **mean squared error** as the training loss:

$$
\text{loss}(\theta) = \frac{1}{N}\sum_{i=1}^{N} \Big(\text{score}(x_i, \theta) - (1 - 2y_i)\Big)^2
$$

Training means finding the $\theta$ that makes this loss as small as possible.

## 🔁 Experiments


Let's train it. We only have one parameter, so a classical optimizer can search for
the best value very quickly.


In [ ]:
from scipy.optimize import minimize

def loss(theta):
    scores = np.array([predict_score(x, theta) for x in xs])
    targets = 1 - 2 * ys
    return np.mean((scores - targets) ** 2)

result = minimize(loss, x0=theta_start, method="COBYLA")
theta_trained = result.x
print("Trained parameter:", theta_trained)
print("Final training loss:", result.fun)


In [ ]:
# Visualize the model's decision function before vs. after training
xs_fine = np.linspace(-2.5, 2.5, 200)
scores_before = [predict_score(x, theta_start) for x in xs_fine]
scores_after = [predict_score(x, theta_trained) for x in xs_fine]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs_fine, scores_before, "--", label="before training", color="gray")
ax.plot(xs_fine, scores_after, label="after training", color="crimson")
ax.axhline(0, color="black", linewidth=0.5)
ax.scatter(xs, 1 - 2 * ys, c=ys, cmap="coolwarm", edgecolor="k", zorder=5, label="data (target scale)")
ax.set_xlabel("signal feature value")
ax.set_ylabel("model score")
ax.legend()
ax.set_title("Quantum classifier: before vs. after training")
fig.tight_layout()


The trained curve should cross zero somewhere between the two clusters; that
crossing point is the classifier's learned decision boundary. Everything to one
side scores positive (predicted "known"), everything to the other scores negative
(predicted "unknown").

In [ ]:
# Evaluate accuracy on the training data itself
predicted_labels = np.array([0 if predict_score(x, theta_trained) > 0 else 1 for x in xs])
accuracy = np.mean(predicted_labels == ys)
print(f"Training accuracy: {accuracy:.0%}")


## 🚀 Challenge


**A new signal comes in.**

The telescope just logged a new reading at `x_new = 0.3`. Using the trained model
(don't retrain it. Reuse `theta_trained`), predict whether it's "known" (0) or
"unknown" (1). Then try a couple of other values of `x_new` near the boundary you
saw in the plot above, and see how confident (how far from zero) the score gets as
you move away from that boundary.


In [ ]:
# Your turn — classify x_new = 0.3 using theta_trained, then try a few more values.


## 🪞 Reflection


Be honest with yourself about what just happened: one qubit and one trainable
parameter can only ever learn a fairly simple decision boundary; this toy problem
was built to be separable by exactly that. The value of this notebook isn't the
sophistication of the model, it's the pattern underneath it: **encode data onto
qubits, run a parameterized circuit, compare its output to a target, let a
classical optimizer adjust the quantum parameters, repeat.** That pattern is
exactly what scales up; with more qubits, richer feature maps, and more
parameters; into the Variational Quantum Classifiers and Quantum Kernels in the
full Quantum Machine Learning track.

## 📦 Summary

- Quantum Machine Learning trains a parameterized quantum circuit against labeled data
- Angle encoding is a simple feature map: turn a classical number into a rotation
- A hybrid loop pairs a quantum circuit with a classical optimizer
- Even a single trainable parameter can learn a real (if simple) decision boundary
- This is a preview; the full path scales this exact pattern up considerably

## ➡️ Next Mission


This concludes the eight-notebook foundational release of Ode to Quantum. You've
gone from powering on a simulator to training a working quantum classifier, writing
real Qiskit at every step along the way.

The full curriculum roadmap: quantum algorithms (Deutsch-Jozsa, Grover, QFT),
variational computing (VQE, QAOA), and the complete Quantum Machine Learning track
(feature maps, VQC, quantum kernels, and a capstone project); continues from here,
mapped out and ready to build.


---
*End of transmission.*